In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT MÔI TRƯỜNG
# ==============================================================================
!pip install transformers pyvi pandas scikit-learn openpyxl

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, TensorDataset
from pyvi import ViTokenizer
import json
import os

# ==============================================================================
# BƯỚC 2: ĐỊNH NGHĨA MODEL 1 (ViTHSD)
# ==============================================================================
class Config:
    BERT_NAME = "vinai/phobert-base"
    HIDDEN_SIZE = 768
    NUM_CLASSES = 4
    GRU_HIDDEN = 128
    CNN_FILTERS = 64
    CNN_KERNEL_SIZES = [2, 3, 4]

class PhoBertHybridModel(nn.Module):
    def __init__(self, config):
        super(PhoBertHybridModel, self).__init__()
        self.bert = AutoModel.from_pretrained(config.BERT_NAME)
        self.gru = nn.GRU(input_size=config.HIDDEN_SIZE, hidden_size=config.GRU_HIDDEN, bidirectional=True, batch_first=True)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=config.HIDDEN_SIZE, out_channels=config.CNN_FILTERS, kernel_size=k)
            for k in config.CNN_KERNEL_SIZES
        ])
        concat_dim = (config.GRU_HIDDEN * 2) + (config.CNN_FILTERS * len(config.CNN_KERNEL_SIZES))
        self.dense = nn.Linear(concat_dim, 256)
        self.batch_norm = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.3)
        self.fc_individual = nn.Linear(256, config.NUM_CLASSES)
        self.fc_group = nn.Linear(256, config.NUM_CLASSES)
        self.fc_societal = nn.Linear(256, config.NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0]
        cls_embedding = bert_out[:, 0, :]
        return cls_embedding

# ==============================================================================
# BƯỚC 3: ĐỊNH NGHĨA MODEL 2 HEAD
# ==============================================================================
class TypeAttackHead(nn.Module):
    def __init__(self, input_dim=768, num_classes=19):
        super(TypeAttackHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# ==============================================================================
# BƯỚC 4: QUY TRÌNH HUẤN LUYỆN (ĐÃ SỬA LOGIC MULTI-LABEL)
# ==============================================================================
def train_head_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"⚙️ Device: {device}")

    # --- 1. Load Model 1 ---
    print("📥 Loading Model 1 (ViTHSD)...")
    model1 = PhoBertHybridModel(Config()).to(device)

    if os.path.exists("/content/drive/MyDrive/SE363-Team13/model/final/best_model.pth"):
        model1.load_state_dict(torch.load("/content/drive/MyDrive/SE363-Team13/model/final/best_model.pth", map_location=device))
    elif os.path.exists("best_model.pth"):
        model1.load_state_dict(torch.load("best_model.pth", map_location=device))
    elif os.path.exists("thsd.pth"):
        model1.load_state_dict(torch.load("thsd.pth", map_location=device))
    else:
        print("❌ LỖI: Không tìm thấy file model 1.")
        return

    model1.eval()
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

    # --- 2. Load & Xử lý Dữ liệu (MULTI-LABEL FIX) ---
    print("📂 Loading Data...")
    excel_path = "/content/drive/MyDrive/SE363-Team13/data/19_type_attack_processed.xlsx"
    if not os.path.exists(excel_path):
        excel_path = "19_type_attack_processed.xlsx"

    df = pd.read_excel(excel_path)
    print(f"📊 Tổng số dòng: {len(df)}")

    LABEL_COLUMNS = [
        "Threat", "Scam", "Misinformation", "Boycott",
        "Body Shaming", "Sexual Harassment", "Intelligence", "Moral", "Victim Blaming",
        "Gender", "Regionalism", "Racism", "Classism", "Religion",
        "Politics", "Social Issues", "Product", "Community",
        "Other"
    ]

    if 'attack_type' in df.columns:
        print("✅ Tìm thấy cột 'attack_type'. Bắt đầu xử lý Multi-label...")

        labels_matrix = np.zeros((len(df), len(LABEL_COLUMNS)))
        valid_rows = []
        texts = []

        skipped_count = 0

        for idx, row in df.iterrows():
            text = str(row['cmt_processed']) if pd.notna(row['cmt_processed']) else ""
            label_str = str(row['attack_type']) # Ví dụ: "Politics, Religion"

            if not text.strip():
                skipped_count += 1
                continue

            # --- LOGIC TÁCH NHÃN ĐA DẠNG ---
            # Tách chuỗi bằng dấu phẩy
            current_labels = [l.strip() for l in label_str.split(',')]

            has_valid_label = False
            for lbl in current_labels:
                if lbl in LABEL_COLUMNS:
                    col_index = LABEL_COLUMNS.index(lbl)
                    labels_matrix[len(valid_rows), col_index] = 1.0
                    has_valid_label = True

            if has_valid_label:
                texts.append(text)
                valid_rows.append(idx)
            else:
                skipped_count += 1

        labels = labels_matrix[:len(valid_rows)]

        print("-" * 30)
        print(f"✅ Số dòng hợp lệ (sau khi tách nhãn): {len(texts)}")
        print(f"⚠️ Số dòng bị bỏ qua (không có nhãn nào khớp): {skipped_count}")
        print("-" * 30)

    else:
        print(f"❌ LỖI: Không tìm thấy cột 'attack_type'. Columns: {df.columns.tolist()}")
        return

    if len(texts) == 0:
        print("❌ LỖI: Không có dữ liệu hợp lệ!")
        return

    # --- 3. Tạo Embeddings ---
    print("🔄 Đang tạo Embeddings (Model 1)...")
    all_embeddings = []

    BATCH_SIZE = 64
    for i in range(0, len(texts), BATCH_SIZE):
        batch_text = texts[i : i+BATCH_SIZE]
        segmented = [ViTokenizer.tokenize(t) for t in batch_text]
        inputs = tokenizer(segmented, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        with torch.no_grad():
            emb = model1(inputs['input_ids'], inputs['attention_mask'])
            all_embeddings.append(emb.cpu())

    X_tensor = torch.cat(all_embeddings)
    y_tensor = torch.tensor(labels, dtype=torch.float32)

    # --- 4. Train Model 2 Head ---
    print("🚀 Training Head...")
    dataset = TensorDataset(X_tensor, y_tensor)

    train_size = int(0.9 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

    head_model = TypeAttackHead(num_classes=len(LABEL_COLUMNS)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(head_model.parameters(), lr=1e-3)

    for epoch in range(20):
        head_model.train()
        total_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            outputs = head_model(bx)
            loss = criterion(outputs, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")

    # --- 5. Lưu Model ---
    print("💾 Saving...")
    torch.save(head_model.state_dict(), "model2_head.pth")
    with open("label_map.json", "w", encoding='utf-8') as f:
        json.dump(LABEL_COLUMNS, f, ensure_ascii=False)

    print("\n🎉 DONE! Files: model2_head.pth, label_map.json")

if __name__ == "__main__":
    train_head_model()


⚙️ Device: cuda
📥 Loading Model 1 (ViTHSD)...
📂 Loading Data...
📊 Tổng số dòng: 3693
✅ Tìm thấy cột 'attack_type'. Bắt đầu xử lý Multi-label...
------------------------------
✅ Số dòng hợp lệ (sau khi tách nhãn): 3692
⚠️ Số dòng bị bỏ qua (không có nhãn nào khớp): 1
------------------------------
🔄 Đang tạo Embeddings (Model 1)...
🚀 Training Head...
   Epoch 5: Loss = 0.1554
   Epoch 10: Loss = 0.1164
   Epoch 15: Loss = 0.0901
   Epoch 20: Loss = 0.0679
💾 Saving...

🎉 DONE! Files: model2_head.pth, label_map.json
